# Chapter 5 — Paradigms That Already Work

**Book:** *Brain–Computer Interfaces and Cognitive Computing: An Introduction*  
**Prerequisites:** Chapters 0–4.  
**Author:** Er. Rishabh Aryan

Chapter 4 named the loop. This chapter fills Stages S0–S5 with four families that have survived contact with users: sensorimotor rhythms, P300 and related ERPs, steady-state and coded evoked potentials, and attempted speech. Each family is taught as a *design object*: generating physiology, canonical experiment, what is decoded, where latency lives, and how the paradigm fails.

New names appear every year. A first-semester course that chases them will never reach evaluation. Learn these four well.

**Learning outcomes**

1. Explain the neural rationale of each family in the vocabulary of Chapter 2.
2. Match a named user and purpose to a paradigm and defend the match.
3. Locate, for each family, the dominant term in the latency budget of Chapter 4.
4. State typical failure modes, including BCI illiteracy, visual fatigue, and non-stationarity.
5. Describe attempted-speech decoding as an extension of the same loop, not as a different science.


---

## 5.1 How to read a paradigm

For every family below, answer the same eight questions before you admire a figure.

1. What in tissue is supposed to change? (S0)  
2. How is that change elicited — active, reactive, or passive?  
3. What sensor is sufficient in principle? (S1)  
4. What feature is the decoder allowed to see? (S4)  
5. What is the output alphabet? Class, character, kinematics, linguistic unit? (S5)  
6. What prior at S6 is usually smuggled in?  
7. What dominates latency?  
8. Who cannot use it?

If a paper does not let you answer these, it has described a result, not a paradigm.

> **Definition 5.1 (BCI paradigm).**  
> A repeatable pairing of a neural generating mechanism, an experimental protocol that elicits it, and a decoder family matched to that mechanism.


---

## 5.2 Sensorimotor rhythms and motor imagery

### Generating physiology

Over sensorimotor cortex, populations generate **mu** (roughly 8–13 Hz) and **beta** (roughly 13–30 Hz) rhythms that desynchronise with actual movement, with movement preparation, and — in many users — with **motor imagery**. The scalp sees that desynchronisation as a drop in band power (ERD). A rebound (ERS) can follow.

The effect is a population statistic (Chapter 2). It is not a spike code for “left hand” painted onto electrode C3.

### Protocol

The user is cued to imagine opening a hand, moving the feet, or performing another kinesthetically vivid action, or to rest. Trials last on the order of seconds. Feedback may be a cursor or a bar. Training across sessions is expected: co-adaptation is the rule (Definition 4.5).

This is an **active** paradigm. The user must generate the covert action.

### Features and decoder

Classic pipeline: band-pass in mu/beta, spatial filter (often common spatial patterns, Chapter 6), log-variance or power, linear classifier. Continuous control maps a score to cursor velocity rather than to a single trial label.

### Latency

Dominated by the analysis window needed to estimate power — often 0.5–2 s — plus the user’s own sluggishness at switching imagery. Model compute is rarely the bottleneck.

### Who it serves

Cursor control, simple device switches, some rehabilitation loops that need an “intent to move” marker (Chapter 10).

### How it fails

- **BCI illiteracy / inefficiency.** A substantial fraction of users never produce a stable ERD the decoder can use. Do not treat this as moral failure. Treat it as a reason to change paradigm.
- **Artefact policy.** Jaw, neck, and forehead EMG occupy overlapping bands. Without an artefact audit (Chapter 4), you may have built a muscle switch.
- **Non-stationarity.** Monday morning $\theta$ is a hypothesis about Monday afternoon.
- **Impoverished alphabet.** Two or three classes are typical at the scalp. That is a switchboard, not a keyboard.

### Amina

If she can still produce imagery and will tolerate training, motor imagery can give a binary switch. It is usually the wrong family for conversation. If she cannot reliably imagine movement, this family is finished before the classifier is written.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(5)
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

fs = 256
t = np.arange(0, 4.0, 1 / fs)
# cue at 1 s; imagery 1-3 s
mu = np.ones_like(t)
mu[(t >= 1.0) & (t < 3.0)] = 0.35
sig = mu * np.sin(2 * np.pi * 10 * t) + 0.2 * np.sin(2 * np.pi * 20 * t) + 0.25 * np.random.randn(t.size)


def rolling_band_power(x, fs, band=(8, 13), win=0.75):
    nwin = int(win * fs)
    hop = int(0.1 * fs)
    freqs = np.fft.rfftfreq(nwin, 1 / fs)
    mask = (freqs >= band[0]) & (freqs < band[1])
    times, pows = [], []
    for start in range(0, len(x) - nwin, hop):
        seg = x[start:start + nwin] * np.hanning(nwin)
        spec = np.abs(np.fft.rfft(seg)) ** 2
        times.append((start + nwin) / fs)  # causal: time at window end
        pows.append(spec[mask].mean())
    return np.array(times), np.array(pows)


tp, pw = rolling_band_power(sig, fs)

fig, axes = plt.subplots(2, 1, figsize=(7.6, 4.4), sharex=True)
axes[0].plot(t, sig, color="steelblue", linewidth=0.6)
axes[0].axvspan(1, 3, color="orange", alpha=0.15, label="imagery cue window")
axes[0].set_ylabel("amplitude")
axes[0].set_title("Cartoon C3-like trace: mu attenuated during imagery")
axes[0].legend(loc="upper right")
axes[1].plot(tp, pw, color="darkorange", linewidth=1.4)
axes[1].set_xlabel("time (s), causal window end")
axes[1].set_ylabel("8–13 Hz power")
axes[1].set_title("Feature the decoder actually sees (delayed by the window)")
plt.tight_layout()
plt.show()


Notice the feature trace turns down *after* imagery starts. That delay is Definition 4.4 made visible. Shortening the window reduces delay and inflates variance. That trade-off *is* the motor-imagery design problem at S4.


---

## 5.3 P300 and related event-related potentials

### Generating physiology

When a rare, task-relevant stimulus appears in a stream, a positive deflection around 300 ms post-stimulus — the **P300** — can be measured at midline parietal scalp. It is a population ERP, variable across trials, small relative to ongoing EEG. Averaging or statistical pooling across repetitions reveals it.

Related families use other ERPs (for example motion-onset, error-related potentials). The loop logic is the same: time-lock, pool, decide.

### Protocol

The classic **Farwell–Donchin matrix**: rows and columns flash. The user attends to one character. Flashes that include that character are rares. Repeat the sequence until the decoder’s posterior over characters is usable. This is **reactive**. The user must see (or in auditory variants, hear) the stimuli.

### Features and decoder

Epochs locked to flash onsets. Features: samples, windows, or spatial projections of the ERP. Classifier scores oddball versus frequent flashes. Scores accumulate over repetitions. A language model at S6 can reweight characters given the prefix already spelled (Chapter 0, Bayes).

### Latency

Dominated by **repetitions**, not by $f_\theta$. If each sequence pass lasts two seconds and you need five passes, you have spent ten seconds before a character is issued, plus any undo. That can still beat silence. It will not beat a reliable eye-tracker.

### Who it serves

Communication when motor imagery is unavailable and vision (or a designed auditory analogue) is available. Binary P300 variants exist for yes/no.

### How it fails

- Visual fatigue and photosensitive risk.
- Gaze dependence: some “P300” systems secretly need the eyes to point. If so, compare against a gaze speller (Chapter 1).
- Need for many repetitions in users with small ERPs.
- Language-model credit: S6 must be reported or English will be attributed to S5.

### Amina

If she can fixate and tolerate flashing, a P300 matrix is a standard first communication loop. If she cannot see, an auditory oddball is a different experiment, not a trivial port.


In [ ]:
# Cartoon oddball: rares carry a late positive bump.
fs = 256
t = np.arange(0, 0.8, 1 / fs)
n_freq, n_rare = 40, 8


def erp(kind):
    noise = 8.0 * np.random.randn(t.size)
    if kind == "rare":
        bump = 6.0 * np.exp(-0.5 * ((t - 0.32) / 0.05) ** 2)
        return noise + bump
    return noise


freq = np.stack([erp("freq") for _ in range(n_freq)])
rare = np.stack([erp("rare") for _ in range(n_rare)])

fig, axes = plt.subplots(1, 2, figsize=(7.8, 3.2), sharey=True)
axes[0].plot(t, freq.T, color="steelblue", alpha=0.15, linewidth=0.7)
axes[0].plot(t, freq.mean(0), color="navy", linewidth=2.0, label="mean frequent")
axes[0].set_title("Frequent flashes")
axes[0].set_xlabel("time from flash (s)")
axes[0].set_ylabel("amplitude (a.u.)")
axes[0].legend()
axes[1].plot(t, rare.T, color="darkorange", alpha=0.25, linewidth=0.7)
axes[1].plot(t, rare.mean(0), color="sienna", linewidth=2.0, label="mean rare")
axes[1].set_title("Rare (attended) flashes")
axes[1].set_xlabel("time from flash (s)")
axes[1].legend()
plt.tight_layout()
plt.show()
print("single-trial SNR is poor; the paradigm spends time pooling repetitions")


---

## 5.4 Steady-state and coded evoked potentials

### Generating physiology

A visual stimulus flickering at frequency $f$ can lock cortical populations, especially over posterior cortex, producing a **steady-state visually evoked potential (SSVEP)** at $f$ and harmonics. Related methods use **code-modulated VEP (cVEP)**: each target carries a distinct temporal code rather than a single frequency.

Auditory and tactile analogues exist. They are harder and slower. Do not advertise a visual method as “gaze independent” without testing covert attention.

### Protocol

Targets on a screen flicker at different frequencies or codes. The user attends to one. The decoder asks which template the recording matches. **Reactive.**

### Features and decoder

Narrow-band power at stimulus frequencies; canonical correlation against flicker references; template correlation for codes. Spatial filters help. The alphabet size can be larger than motor imagery because the experimenter, not the user, generates the labels in the world.

### Latency

A decision can be issued after a fraction of a second to a few seconds of flicker, depending on how small an error you will tolerate. Faster than typical P300 matrices; still bound by window length and by visual tolerance.

### Who it serves

High-rate selection when vision is good and flicker is acceptable. Research and some communication prototypes.

### How it fails

- Flicker fatigue, migraine, photosensitive epilepsy — exclusion criteria are part of the paradigm.
- Gaze confound: frequency tagging often tracks where the eyes point. Then you must beat an eye-tracker, not a P300 system.
- Lighting and display refresh constrain available frequencies.
- Not a speech channel.

### Amina

Use only if she can look at a stable display and flicker is medically acceptable. Otherwise this family is closed.


In [ ]:
fs = 256
t = np.arange(0, 2.0, 1 / fs)
f1, f2 = 8.0, 13.0
# attended target at 8 Hz, plus noise and a little 13 Hz leak
eeg = 1.2 * np.sin(2 * np.pi * f1 * t) + 0.2 * np.sin(2 * np.pi * f2 * t) + 0.5 * np.random.randn(t.size)
freqs = np.fft.rfftfreq(t.size, 1 / fs)
psd = np.abs(np.fft.rfft(eeg * np.hanning(t.size))) ** 2

fig, axes = plt.subplots(1, 2, figsize=(7.8, 3.2))
axes[0].plot(t[: fs * 1], eeg[: fs * 1], color="steelblue", linewidth=0.8)
axes[0].set_title("Cartoon posterior trace (attended 8 Hz)")
axes[0].set_xlabel("time (s)")
axes[0].set_ylabel("amplitude")
axes[1].plot(freqs[(freqs > 2) & (freqs < 30)], psd[(freqs > 2) & (freqs < 30)], color="sienna")
axes[1].axvline(f1, color="black", linestyle="--", linewidth=1, label="8 Hz")
axes[1].axvline(f2, color="gray", linestyle=":", linewidth=1, label="13 Hz")
axes[1].set_title("Power spectrum")
axes[1].set_xlabel("Hz")
axes[1].legend()
plt.tight_layout()
plt.show()


---

## 5.5 Attempted speech and related high-bandwidth cortical decoding

### Generating physiology

Attempting to speak — even without audible output — engages population activity in ventral motor, premotor, and related speech areas. With sufficiently fine sensors (ECoG or intracortical arrays; Chapter 3), those patterns can be mapped onto articulatory features, phonemes, words, or characters. **Inner speech** without attempt is a different, weaker, and less settled target. Do not conflate the two in a methods sentence.

### Protocol

The user is cued to attempt a sentence, or to speak silently in a constrained vocabulary, while cortical signals are recorded. Training pairs neural windows with text or with audio. Closed-loop use shows the decoded words so the user can continue or repair. **Active**, high invasiveness in the systems that currently work.

### Features and decoder

Short time windows of multi-channel activity; often learned representations rather than hand-crafted bands; sequence models whose output is linguistic. Stage S6 is no longer optional in spirit: a language model is a prior over the next unit (Definition 1.5 in use). Credit assignment between S5 and S6 is a scientific duty.

### Latency

Conversational constraint. A delay of several seconds turns speech into dictation. Window hop, sequence context, and synthesis buffering all appear in the budget of Section 4.4.

### Who it serves

People for whom scalp spellers are too slow and surgery is accepted inside a supported trial. This is Amina’s later loop, not her first.

### How it fails

- Surgical and chronic-device cost (Definitions 3.1 and 3.4).
- Vocabulary and language mismatch: a model trained on English sentences is not a universal speech organ.
- Inner-speech overclaim.
- Stability of units and of the mapping across weeks.
- Evaluation that reports next-unit accuracy without reporting whether a listener understood the utterance.

### Relation to the other families

The loop is the same loop. What changed is S1 grain and the output alphabet. Students who treat speech BCI as a different science will not notice when S6 is doing the linguistic work.


---

## 5.6 Choosing a family: a decision table

| Question | If yes, consider | If no |
|---|---|---|
| Can the user look at flicker safely? | SSVEP / cVEP | P300 or non-visual |
| Can the user attend rares but not tolerate flicker? | P300 | imagery or implant path |
| Is a 2–3 class switch enough? | Motor imagery or binary ERP | linguistic families |
| Is conversation the goal and surgery accepted? | Attempted speech | scalp spellers first |
| Is residual gaze reliable? | Test an eye-tracker before any BCI | BCI may be justified |
| Is the user in rehabilitation, not conversation? | Imagery or ERP as a trigger in a therapy loop | do not sell bit-rate |

> **Definition 5.2 (Paradigm mismatch).**  
> Selection of a family whose generating mechanism the user cannot produce, whose sensor the institution cannot support, or whose alphabet cannot serve the declared purpose.

Most failed student projects are Definition 5.2, not failed neural networks.


---

## 5.7 Hybrids and extras (keep them secondary)

**Hybrid BCI.** Combine two families, or a BCI with EMG, gaze, or a switch. Legitimate when disclosed. A P300 plus a residual blink key may be the system Amina will actually use.

**Error-related potentials.** A deflection after the user notices a mistake can support an undo. Treat as an ERP family sitting at S6 as a correction channel.

**Passive states.** Workload and fatigue estimators (Chapter 11) can wrap any of the four families as policy. They are not a fifth communication alphabet.

Do not start a first project on a hybrid of four methods. Start on one family and add a switch only when the family is measured.


---

## 5.8 What to carry into Chapter 6

Chapter 6 opens Stage S3–S4: how traces become features. Carry four sentences.

1. Name the family before you name the filter.
2. Motor imagery lives in band power and spatial patterns; P300 lives in time-locked shape; SSVEP lives in narrow-band locking; speech lives in multi-channel sequences.
3. Latency is usually repetitions or windows, not the classifier.
4. If the user cannot produce the generating mechanism, no later chapter will help.


---

## 5.9 Chapter summary

Four families currently carry the field’s working load. Sensorimotor rhythms support active, low-alphabet control and some therapy loops. P300 and related ERPs support reactive communication by pooling weak time-locked responses. SSVEP and coded VEP support faster reactive selection when flicker is acceptable and gaze confounds are faced. Attempted-speech decoding extends the same loop to a linguistic alphabet at higher surgical cost, with language models as priors that must be credited. Choose a family by user and purpose. A mismatch of paradigm is a deeper error than a mismatch of classifier.


---

## Exercises

**5.1** Answer the eight questions of Section 5.1 for a 6 × 6 P300 matrix used by Amina at home.

**5.2** A paper reports a “mind-typing SSVEP system” at 12 characters per minute. List three facts you require before you accept that the system is a BCI rather than an eye-tracker with an EEG hat.

**5.3** Why is motor-imagery “illiteracy” not solved by a deeper network trained on the same user’s non-informative scalp rhythms? One paragraph.

**5.4** Using the rolling-power cell, estimate how much later the mu-power feature responds if the window is 1.5 s instead of 0.75 s. Relate the answer to Definition 4.4.

**5.5** Credit assignment: an attempted-speech system emits grammatical English. Describe an ablation that would show how much of that grammar is S5 versus S6.

**5.6** Amina refuses flicker and cannot fixate a matrix. Which families remain? What comparison device must still be tested?

**5.7** Design, on one page, a hybrid that adds a residual-muscle undo key to a P300 speller. Mark the stage of every component. State how you will stop the muscle key from being reported as neural accuracy.

**5.8** Inner speech versus attempted speech: write six sentences a journalist would be allowed to publish after reading this chapter.


---

## Annotated reading

- Farwell and Donchin (1988) — the matrix that still structures reactive communication.  
- Wolpaw’s sensorimotor-rhythm programme — read for training and co-adaptation, not only for online rates.  
- A current SSVEP / cVEP methods review — insist on the gaze discussion.  
- A contemporary attempted-speech closed-loop paper; mark S5 and S6 with two different colours before you quote the headline rate.

---

*End of Chapter 5.*  
*Next: Chapter 6 — From traces to features.*
